# Chapter 7.3: LLM Agents for Recommendation

## Learning Objectives

By the end of this notebook, you will be able to:

1. Understand the architecture of LLM-powered recommendation agents
2. Implement a RecAgent-style recommendation agent with planning and reasoning
3. Build an InteRecAgent with tool-calling capabilities
4. Design memory systems for recommendation agents (short-term and long-term)
5. Implement reflection mechanisms for agent self-improvement
6. Create a complete LLM rec agent pipeline with simulated tool calls
7. Evaluate LLM agent recommendation quality

## Prerequisites

- Understanding of LLMs and prompt engineering
- Basic recommendation system concepts
- Familiarity with Python classes and design patterns

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USERNAME/rec_system/blob/main/notebooks/part7/chapter_7.3_llm_agents.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://github.com/USERNAME/rec_system/raw/main/notebooks/part7/chapter_7.3_llm_agents.ipynb)

In [ ]:
import numpy as np
import json
import re
import random
import matplotlib.pyplot as plt
from typing import List, Dict, Optional, Tuple, Any
from collections import defaultdict
from dataclasses import dataclass, field

np.random.seed(42)
random.seed(42)

plt.style.use('seaborn-v0_8')
print("All imports successful.")

## 1. LLM Agents for Recommendation: Overview

Recent work has explored using Large Language Models as recommendation agents that can:

1. **Reason** about user preferences through natural language
2. **Plan** multi-step recommendation strategies
3. **Use tools** to access external information
4. **Remember** past interactions and user preferences
5. **Reflect** on recommendation quality and improve

Key papers:
- **RecAgent** (Wang et al., 2023): LLM-powered agent that simulates user behavior
- **InteRecAgent** (Huang et al., 2023): Interactive rec agent with tool-calling
- **RecMind** (Wang et al., 2023): Planning-based recommendation agent

> **💡 Concept:** Unlike traditional rec systems that predict scores, LLM agents can engage in **multi-turn dialogue**, ask clarifying questions, explain reasoning, and adapt strategies dynamically. They represent a paradigm shift from prediction to conversation.

Since this notebook runs without external APIs, we simulate LLM behavior with rule-based agents that demonstrate the same architectural patterns.

In [ ]:
# Synthetic item catalog
@dataclass
class Item:
    item_id: int
    name: str
    category: str
    price: float
    rating: float
    tags: List[str]
    description: str

@dataclass
class UserProfile:
    user_id: int
    name: str
    preferences: List[str]
    budget: float
    history: List[int] = field(default_factory=list)
    ratings: Dict[int, float] = field(default_factory=dict)


def create_synthetic_catalog(n_items: int = 50) -> List[Item]:
    """Create a synthetic product catalog."""
    categories = ['Electronics', 'Books', 'Clothing', 'Home', 'Sports']
    tag_pool = {
        'Electronics': ['wireless', 'portable', 'smart', 'bluetooth', 'USB-C', 'high-res'],
        'Books': ['fiction', 'non-fiction', 'sci-fi', 'mystery', 'self-help', 'technical'],
        'Clothing': ['casual', 'formal', 'outdoor', 'cotton', 'waterproof', 'lightweight'],
        'Home': ['kitchen', 'decor', 'storage', 'eco-friendly', 'compact', 'durable'],
        'Sports': ['fitness', 'outdoor', 'yoga', 'running', 'cycling', 'training'],
    }
    
    items = []
    rng = np.random.RandomState(42)
    
    for i in range(n_items):
        cat = categories[i % len(categories)]
        tags = list(rng.choice(tag_pool[cat], size=rng.randint(2, 4), replace=False))
        price = round(rng.uniform(10, 200), 2)
        rating = round(rng.uniform(3.0, 5.0), 1)
        
        items.append(Item(
            item_id=i,
            name=f"{cat} Item {i}",
            category=cat,
            price=price,
            rating=rating,
            tags=tags,
            description=f"A {rating}-star {cat.lower()} product with features: {', '.join(tags)}. Priced at ${price}."
        ))
    
    return items


def create_synthetic_users(n_users: int = 10) -> List[UserProfile]:
    """Create synthetic user profiles."""
    preference_pools = [
        ['Electronics', 'smart', 'portable'],
        ['Books', 'sci-fi', 'technical'],
        ['Clothing', 'casual', 'outdoor'],
        ['Home', 'eco-friendly', 'kitchen'],
        ['Sports', 'fitness', 'running'],
    ]
    
    users = []
    rng = np.random.RandomState(42)
    
    for i in range(n_users):
        prefs = preference_pools[i % len(preference_pools)]
        budget = round(rng.uniform(50, 300), 2)
        users.append(UserProfile(
            user_id=i,
            name=f"User_{i}",
            preferences=prefs,
            budget=budget
        ))
    
    return users


catalog = create_synthetic_catalog(50)
users = create_synthetic_users(10)
print(f"Catalog: {len(catalog)} items")
print(f"Users: {len(users)} users")
print(f"\nSample item: {catalog[0]}")
print(f"\nSample user: {users[0]}")

## 2. Memory Systems for Rec Agents

LLM agents use two types of memory:

1. **Short-term memory**: Current conversation context, recent interactions
2. **Long-term memory**: Persistent user profile, preference summaries, past recommendations

$$\text{Memory}(t) = \{\underbrace{\text{Context}(t)}_\text{short-term}, \underbrace{\text{Profile}(u), \text{History}(u)}_\text{long-term}\}$$

> **🔑 Pro Tip:** Effective memory management is crucial because LLMs have limited context windows. Summarization, retrieval, and prioritization of memories are active research areas.

In [ ]:
class ShortTermMemory:
    """Stores recent conversation turns and interaction context."""
    
    def __init__(self, max_turns: int = 10):
        self.max_turns = max_turns
        self.turns: List[Dict[str, str]] = []
    
    def add_turn(self, role: str, content: str):
        self.turns.append({'role': role, 'content': content})
        if len(self.turns) > self.max_turns:
            self.turns = self.turns[-self.max_turns:]
    
    def get_context(self) -> str:
        return "\n".join([f"{t['role']}: {t['content']}" for t in self.turns])
    
    def clear(self):
        self.turns = []


class LongTermMemory:
    """Persistent user preference and interaction memory."""
    
    def __init__(self):
        self.user_preferences: Dict[str, float] = {}  # tag -> score
        self.category_preferences: Dict[str, float] = {}  # category -> score
        self.interaction_history: List[Dict] = []
        self.preference_summary: str = ""
    
    def update_from_interaction(self, item: Item, rating: float):
        """Update long-term memory from a new interaction."""
        # Update tag preferences
        for tag in item.tags:
            if tag not in self.user_preferences:
                self.user_preferences[tag] = 0.0
            # Exponential moving average
            alpha = 0.3
            normalized_rating = (rating - 3.0) / 2.0  # Map [1,5] to [-1,1]
            self.user_preferences[tag] = (
                (1 - alpha) * self.user_preferences[tag] + alpha * normalized_rating
            )
        
        # Update category preferences
        cat = item.category
        if cat not in self.category_preferences:
            self.category_preferences[cat] = 0.0
        normalized_rating = (rating - 3.0) / 2.0
        self.category_preferences[cat] = (
            (1 - 0.3) * self.category_preferences[cat] + 0.3 * normalized_rating
        )
        
        # Store interaction
        self.interaction_history.append({
            'item_id': item.item_id,
            'item_name': item.name,
            'category': item.category,
            'rating': rating
        })
        
        # Update summary
        self._update_summary()
    
    def _update_summary(self):
        """Generate a natural language summary of user preferences."""
        liked_tags = sorted(self.user_preferences.items(), key=lambda x: -x[1])[:5]
        liked_cats = sorted(self.category_preferences.items(), key=lambda x: -x[1])[:3]
        
        parts = []
        if liked_cats:
            cats = [f"{c}" for c, s in liked_cats if s > 0]
            if cats:
                parts.append(f"Preferred categories: {', '.join(cats)}")
        if liked_tags:
            tags = [f"{t}" for t, s in liked_tags if s > 0]
            if tags:
                parts.append(f"Liked features: {', '.join(tags)}")
        
        n_interactions = len(self.interaction_history)
        if n_interactions > 0:
            avg_rating = np.mean([h['rating'] for h in self.interaction_history])
            parts.append(f"Total interactions: {n_interactions}, avg rating: {avg_rating:.1f}")
        
        self.preference_summary = ". ".join(parts)
    
    def get_summary(self) -> str:
        return self.preference_summary if self.preference_summary else "No interactions yet."


# Demonstrate memory system
ltm = LongTermMemory()
for item_id in [0, 5, 10, 15, 20]:
    rating = np.random.choice([3.0, 4.0, 4.5, 5.0])
    ltm.update_from_interaction(catalog[item_id], rating)

print("Long-term Memory Summary:")
print(ltm.get_summary())
print(f"\nTag preferences: {dict(sorted(ltm.user_preferences.items(), key=lambda x: -x[1])[:5])}")

## 3. RecAgent: LLM-Powered Recommendation Agent

A RecAgent follows the **Observe -> Think -> Act** loop:

1. **Observe**: Gather user context, history, current request
2. **Think**: Reason about preferences, plan recommendation strategy
3. **Act**: Generate recommendations with explanations

We simulate this with rule-based reasoning that mirrors LLM agent behavior.

In [ ]:
class RecAgent:
    """LLM-powered recommendation agent (simulated).
    
    Implements the Observe-Think-Act loop for recommendation.
    Based on RecAgent (Wang et al., 2023) architecture.
    """
    
    def __init__(self, catalog: List[Item]):
        self.catalog = catalog
        self.item_index = {item.item_id: item for item in catalog}
        self.short_memory = ShortTermMemory(max_turns=10)
        self.long_memory = LongTermMemory()
        self.reasoning_trace: List[str] = []
    
    def observe(self, user: UserProfile, query: str) -> Dict[str, Any]:
        """Gather and organize all available context."""
        observation = {
            'user_name': user.name,
            'stated_preferences': user.preferences,
            'budget': user.budget,
            'history': user.history,
            'query': query,
            'conversation': self.short_memory.get_context(),
            'long_term_summary': self.long_memory.get_summary(),
        }
        self.reasoning_trace.append(f"[OBSERVE] User: {user.name}, Query: '{query}'")
        return observation
    
    def think(self, observation: Dict[str, Any]) -> Dict[str, Any]:
        """Reason about the recommendation strategy."""
        plan = {
            'strategy': '',
            'filters': {},
            'ranking_criteria': [],
            'reasoning': ''
        }
        
        query = observation['query'].lower()
        prefs = observation['stated_preferences']
        budget = observation['budget']
        
        # Determine recommendation strategy
        if 'similar' in query or 'like' in query:
            plan['strategy'] = 'content_based'
            plan['reasoning'] = "User wants items similar to past preferences. Using content-based filtering."
        elif 'new' in query or 'different' in query or 'explore' in query:
            plan['strategy'] = 'exploration'
            plan['reasoning'] = "User wants to explore new items. Will diversify recommendations."
        elif 'budget' in query or 'cheap' in query or 'affordable' in query:
            plan['strategy'] = 'budget_constrained'
            plan['filters']['max_price'] = budget * 0.5
            plan['reasoning'] = f"Budget-conscious query. Filtering items under ${budget * 0.5:.0f}."
        elif 'best' in query or 'top' in query:
            plan['strategy'] = 'top_rated'
            plan['ranking_criteria'] = ['rating', 'relevance']
            plan['reasoning'] = "User wants best items. Ranking by rating and relevance."
        else:
            plan['strategy'] = 'hybrid'
            plan['reasoning'] = "General query. Using hybrid strategy (relevance + diversity)."
        
        # Add preference-based filters
        plan['filters']['preferred_tags'] = prefs
        plan['filters']['max_price'] = plan['filters'].get('max_price', budget)
        
        self.reasoning_trace.append(f"[THINK] Strategy: {plan['strategy']}. {plan['reasoning']}")
        return plan
    
    def act(self, observation: Dict[str, Any], plan: Dict[str, Any], 
            top_k: int = 5) -> List[Dict]:
        """Generate recommendations based on the plan."""
        candidates = self._filter_candidates(plan['filters'], observation['history'])
        scored = self._score_candidates(candidates, observation, plan)
        
        if plan['strategy'] == 'exploration':
            scored = self._add_diversity(scored)
        
        recommendations = sorted(scored, key=lambda x: -x['score'])[:top_k]
        
        # Generate explanations
        for rec in recommendations:
            rec['explanation'] = self._generate_explanation(rec, observation, plan)
        
        self.reasoning_trace.append(
            f"[ACT] Recommended {len(recommendations)} items: "
            f"{[r['item'].name for r in recommendations]}"
        )
        
        # Update short-term memory
        self.short_memory.add_turn('user', observation['query'])
        rec_text = ", ".join([r['item'].name for r in recommendations])
        self.short_memory.add_turn('agent', f"Recommended: {rec_text}")
        
        return recommendations
    
    def _filter_candidates(self, filters: Dict, history: List[int]) -> List[Item]:
        """Filter catalog based on constraints."""
        candidates = []
        max_price = filters.get('max_price', float('inf'))
        
        for item in self.catalog:
            if item.item_id in history:
                continue  # Skip already seen items
            if item.price > max_price:
                continue
            candidates.append(item)
        
        return candidates
    
    def _score_candidates(self, candidates: List[Item], 
                          observation: Dict, plan: Dict) -> List[Dict]:
        """Score candidates based on relevance."""
        prefs = set(observation['stated_preferences'])
        scored = []
        
        for item in candidates:
            # Tag overlap score
            item_tags = set(item.tags + [item.category])
            overlap = len(prefs & item_tags)
            tag_score = overlap / max(len(prefs), 1)
            
            # Rating score (normalized)
            rating_score = (item.rating - 3.0) / 2.0
            
            # Long-term memory score
            ltm_score = 0.0
            for tag in item.tags:
                ltm_score += self.long_memory.user_preferences.get(tag, 0.0)
            ltm_score = max(-1, min(1, ltm_score))  # Clip
            
            # Combined score
            score = 0.4 * tag_score + 0.3 * rating_score + 0.3 * ltm_score
            
            scored.append({
                'item': item,
                'score': score,
                'tag_score': tag_score,
                'rating_score': rating_score,
                'ltm_score': ltm_score
            })
        
        return scored
    
    def _add_diversity(self, scored: List[Dict]) -> List[Dict]:
        """Add diversity bonus for exploration strategy."""
        seen_categories = set()
        for rec in scored:
            cat = rec['item'].category
            if cat in seen_categories:
                rec['score'] *= 0.7  # Penalize same category
            seen_categories.add(cat)
        return scored
    
    def _generate_explanation(self, rec: Dict, observation: Dict, plan: Dict) -> str:
        """Generate a natural language explanation for the recommendation."""
        item = rec['item']
        parts = []
        
        if rec['tag_score'] > 0:
            matching = set(observation['stated_preferences']) & set(item.tags + [item.category])
            parts.append(f"Matches your interest in {', '.join(matching)}")
        
        if item.rating >= 4.5:
            parts.append(f"Highly rated ({item.rating}/5.0)")
        
        if rec['ltm_score'] > 0.2:
            parts.append("Similar to items you've enjoyed before")
        
        if plan['strategy'] == 'budget_constrained':
            parts.append(f"Within your budget at ${item.price}")
        
        return ". ".join(parts) if parts else f"A {item.rating}-star {item.category} item you might enjoy."
    
    def receive_feedback(self, item_id: int, rating: float):
        """Process user feedback to update long-term memory."""
        item = self.item_index[item_id]
        self.long_memory.update_from_interaction(item, rating)
        self.reasoning_trace.append(
            f"[FEEDBACK] User rated {item.name}: {rating}/5.0"
        )


# Create agent and test
agent = RecAgent(catalog)
user = users[0]

# Simulate some history
for item_id in [0, 10, 20]:
    agent.receive_feedback(item_id, np.random.choice([4.0, 4.5, 5.0]))
    user.history.append(item_id)

# Run recommendation
query = "I'm looking for something similar to what I liked before"
obs = agent.observe(user, query)
plan = agent.think(obs)
recs = agent.act(obs, plan, top_k=5)

print("=" * 60)
print(f"Query: {query}")
print(f"Strategy: {plan['strategy']}")
print(f"Reasoning: {plan['reasoning']}")
print("\nRecommendations:")
for i, rec in enumerate(recs, 1):
    item = rec['item']
    print(f"  {i}. {item.name} (${item.price}, {item.rating}/5.0) - Score: {rec['score']:.3f}")
    print(f"     {rec['explanation']}")

## 4. InteRecAgent: Interactive Agent with Tools

InteRecAgent (Huang et al., 2023) extends the basic agent with **tool-calling** capabilities. The agent can:

1. **Search** the catalog with complex queries
2. **Filter** items by attributes
3. **Compare** items side-by-side
4. **Look up** detailed information
5. **Calculate** price comparisons or budgets

> **💡 Concept:** Tools give the agent access to structured data operations that pure LLM reasoning cannot reliably perform (e.g., exact price comparisons, database queries). This is a form of **grounding** the agent in factual data.

In [ ]:
class Tool:
    """Base class for agent tools."""
    def __init__(self, name: str, description: str):
        self.name = name
        self.description = description
    
    def execute(self, **kwargs) -> str:
        raise NotImplementedError


class SearchTool(Tool):
    """Search catalog by keywords."""
    def __init__(self, catalog: List[Item]):
        super().__init__('search', 'Search catalog by category, tags, or keywords')
        self.catalog = catalog
    
    def execute(self, query: str = "", category: str = "", 
                tags: List[str] = None, max_results: int = 10) -> str:
        results = []
        for item in self.catalog:
            score = 0
            if category and item.category.lower() == category.lower():
                score += 2
            if tags:
                overlap = len(set(tags) & set(item.tags))
                score += overlap
            if query:
                if query.lower() in item.description.lower():
                    score += 1
            if score > 0:
                results.append((item, score))
        
        results.sort(key=lambda x: -x[1])
        results = results[:max_results]
        
        if not results:
            return "No items found matching the query."
        
        output = f"Found {len(results)} items:\n"
        for item, score in results:
            output += f"  - {item.name}: ${item.price}, {item.rating}/5.0, tags={item.tags}\n"
        return output


class FilterTool(Tool):
    """Filter items by price range, rating, etc."""
    def __init__(self, catalog: List[Item]):
        super().__init__('filter', 'Filter items by price range, minimum rating, etc.')
        self.catalog = catalog
    
    def execute(self, min_price: float = 0, max_price: float = 1000,
                min_rating: float = 0, category: str = "") -> str:
        results = []
        for item in self.catalog:
            if item.price < min_price or item.price > max_price:
                continue
            if item.rating < min_rating:
                continue
            if category and item.category.lower() != category.lower():
                continue
            results.append(item)
        
        if not results:
            return "No items match the filter criteria."
        
        output = f"Filtered {len(results)} items:\n"
        for item in results[:10]:
            output += f"  - {item.name}: ${item.price}, {item.rating}/5.0\n"
        return output


class CompareTool(Tool):
    """Compare items side by side."""
    def __init__(self, catalog: List[Item]):
        super().__init__('compare', 'Compare two or more items on all attributes')
        self.item_index = {item.item_id: item for item in catalog}
    
    def execute(self, item_ids: List[int] = None) -> str:
        if not item_ids or len(item_ids) < 2:
            return "Need at least 2 item IDs to compare."
        
        items = [self.item_index[i] for i in item_ids if i in self.item_index]
        if len(items) < 2:
            return "Some items not found."
        
        output = "Comparison:\n"
        output += f"{'Attribute':<15}"
        for item in items:
            output += f"{item.name:<25}"
        output += "\n" + "-" * (15 + 25 * len(items)) + "\n"
        
        for attr in ['category', 'price', 'rating', 'tags']:
            output += f"{attr:<15}"
            for item in items:
                val = getattr(item, attr)
                if attr == 'price':
                    val = f"${val}"
                elif attr == 'tags':
                    val = ", ".join(val)
                output += f"{str(val):<25}"
            output += "\n"
        
        return output


class CalculatorTool(Tool):
    """Budget and price calculations."""
    def __init__(self, catalog: List[Item]):
        super().__init__('calculator', 'Calculate totals, savings, or budget remaining')
        self.item_index = {item.item_id: item for item in catalog}
    
    def execute(self, item_ids: List[int] = None, budget: float = 0) -> str:
        if not item_ids:
            return "No items specified."
        
        items = [self.item_index[i] for i in item_ids if i in self.item_index]
        total = sum(item.price for item in items)
        
        output = "Price calculation:\n"
        for item in items:
            output += f"  {item.name}: ${item.price}\n"
        output += f"  Total: ${total:.2f}\n"
        
        if budget > 0:
            remaining = budget - total
            output += f"  Budget: ${budget:.2f}\n"
            output += f"  Remaining: ${remaining:.2f}\n"
            if remaining < 0:
                output += f"  WARNING: Over budget by ${-remaining:.2f}!\n"
        
        return output


# Create tools
tools = {
    'search': SearchTool(catalog),
    'filter': FilterTool(catalog),
    'compare': CompareTool(catalog),
    'calculator': CalculatorTool(catalog),
}

# Demo tool usage
print(tools['search'].execute(category='Electronics', tags=['smart', 'portable']))
print()
print(tools['filter'].execute(min_rating=4.5, max_price=100))
print()
print(tools['compare'].execute(item_ids=[0, 5, 10]))

In [ ]:
class InteRecAgent:
    """Interactive Recommendation Agent with tool-calling.
    
    Based on InteRecAgent (Huang et al., 2023) architecture.
    Uses a plan-execute-reflect loop with tool calling.
    """
    
    def __init__(self, catalog: List[Item], tools: Dict[str, Tool]):
        self.catalog = catalog
        self.tools = tools
        self.short_memory = ShortTermMemory()
        self.long_memory = LongTermMemory()
        self.tool_calls: List[Dict] = []
    
    def process_request(self, user: UserProfile, query: str) -> Dict:
        """Process a user request through plan-execute-reflect."""
        # Step 1: Plan
        tool_plan = self._plan_tool_calls(user, query)
        
        # Step 2: Execute tools
        tool_results = self._execute_tools(tool_plan)
        
        # Step 3: Synthesize results into recommendations
        recommendations = self._synthesize(user, query, tool_results)
        
        # Step 4: Reflect on quality
        reflection = self._reflect(recommendations, user)
        
        return {
            'recommendations': recommendations,
            'tool_calls': self.tool_calls,
            'reflection': reflection
        }
    
    def _plan_tool_calls(self, user: UserProfile, query: str) -> List[Dict]:
        """Plan which tools to call based on the query."""
        plan = []
        query_lower = query.lower()
        
        # Always search based on preferences
        plan.append({
            'tool': 'search',
            'kwargs': {'tags': user.preferences[:3], 'max_results': 10}
        })
        
        # Budget queries need filtering and calculation
        if any(w in query_lower for w in ['budget', 'cheap', 'afford', 'price']):
            plan.append({
                'tool': 'filter',
                'kwargs': {'max_price': user.budget, 'min_rating': 3.5}
            })
        
        # Comparison queries
        if any(w in query_lower for w in ['compare', 'versus', 'vs', 'difference']):
            plan.append({
                'tool': 'compare',
                'kwargs': {'item_ids': [0, 5, 10]}  # Will be refined
            })
        
        # High quality filter for "best" queries
        if any(w in query_lower for w in ['best', 'top', 'highest']):
            plan.append({
                'tool': 'filter',
                'kwargs': {'min_rating': 4.0}
            })
        
        return plan
    
    def _execute_tools(self, plan: List[Dict]) -> List[str]:
        """Execute planned tool calls."""
        results = []
        self.tool_calls = []
        
        for call in plan:
            tool_name = call['tool']
            kwargs = call['kwargs']
            
            if tool_name in self.tools:
                result = self.tools[tool_name].execute(**kwargs)
                results.append(result)
                self.tool_calls.append({
                    'tool': tool_name,
                    'kwargs': kwargs,
                    'result_preview': result[:100] + '...' if len(result) > 100 else result
                })
        
        return results
    
    def _synthesize(self, user: UserProfile, query: str, 
                    tool_results: List[str]) -> List[Dict]:
        """Synthesize tool results into final recommendations."""
        # Parse item mentions from tool results
        mentioned_items = set()
        for result in tool_results:
            for item in self.catalog:
                if item.name in result:
                    mentioned_items.add(item.item_id)
        
        # Score and rank mentioned items
        prefs = set(user.preferences)
        scored = []
        for item_id in mentioned_items:
            item = self.catalog[item_id]
            if item_id in user.history:
                continue
            
            tag_overlap = len(prefs & set(item.tags + [item.category]))
            score = tag_overlap * 0.4 + (item.rating / 5.0) * 0.3
            if item.price <= user.budget:
                score += 0.3
            
            scored.append({
                'item': item,
                'score': score,
                'explanation': f"Matches {tag_overlap} of your preferences. Rated {item.rating}/5.0."
            })
        
        scored.sort(key=lambda x: -x['score'])
        return scored[:5]
    
    def _reflect(self, recommendations: List[Dict], user: UserProfile) -> str:
        """Reflect on recommendation quality."""
        if not recommendations:
            return "Could not find good matches. Consider broadening preferences."
        
        avg_score = np.mean([r['score'] for r in recommendations])
        categories = set(r['item'].category for r in recommendations)
        avg_price = np.mean([r['item'].price for r in recommendations])
        
        reflection = f"Quality check: avg relevance score = {avg_score:.2f}. "
        reflection += f"Category diversity: {len(categories)} unique categories. "
        
        if avg_price > user.budget:
            reflection += f"WARNING: Average price (${avg_price:.0f}) exceeds budget (${user.budget:.0f}). "
        
        if len(categories) < 2:
            reflection += "Low diversity - consider adding items from other categories. "
        
        return reflection


# Test InteRecAgent
inte_agent = InteRecAgent(catalog, tools)
test_user = users[0]

queries = [
    "Show me the best electronics within my budget",
    "I want something new and different to explore",
    "What are the top rated items?"
]

for query in queries:
    print("=" * 60)
    print(f"Query: {query}")
    result = inte_agent.process_request(test_user, query)
    
    print(f"\nTool calls: {len(result['tool_calls'])}")
    for tc in result['tool_calls']:
        print(f"  - {tc['tool']}({tc['kwargs']})")
    
    print(f"\nRecommendations:")
    for i, rec in enumerate(result['recommendations'], 1):
        print(f"  {i}. {rec['item'].name} (${rec['item'].price}) - {rec['explanation']}")
    
    print(f"\nReflection: {result['reflection']}")
    print()

## 5. Reflection and Self-Improvement

A key capability of LLM agents is **reflection** — evaluating their own performance and adjusting strategy.

> **💡 Concept:** Reflection mechanisms allow agents to learn from mistakes within a single session. After receiving user feedback (positive or negative), the agent can adjust its understanding of user preferences and modify its recommendation strategy.

The reflection loop:
1. Generate recommendations
2. Receive user feedback
3. Analyze what went right/wrong
4. Update strategy for next round

In [ ]:
class ReflectiveRecAgent(RecAgent):
    """RecAgent with reflection and self-improvement capabilities."""
    
    def __init__(self, catalog: List[Item]):
        super().__init__(catalog)
        self.reflections: List[str] = []
        self.strategy_adjustments: Dict[str, float] = {
            'diversity_weight': 0.3,
            'rating_weight': 0.3,
            'preference_weight': 0.4,
        }
    
    def reflect_on_feedback(self, recommendations: List[Dict], 
                           feedback: Dict[int, float]) -> str:
        """Reflect on user feedback and adjust strategy."""
        liked = []
        disliked = []
        
        for rec in recommendations:
            item_id = rec['item'].item_id
            if item_id in feedback:
                rating = feedback[item_id]
                if rating >= 4.0:
                    liked.append(rec)
                elif rating <= 2.0:
                    disliked.append(rec)
                # Update long-term memory
                self.receive_feedback(item_id, rating)
        
        # Analyze patterns
        reflection_parts = []
        
        if liked:
            liked_tags = set()
            for r in liked:
                liked_tags.update(r['item'].tags)
            reflection_parts.append(
                f"User liked items with tags: {', '.join(liked_tags)}. "
                f"Increasing preference weight."
            )
            self.strategy_adjustments['preference_weight'] = min(
                0.6, self.strategy_adjustments['preference_weight'] + 0.05
            )
        
        if disliked:
            disliked_tags = set()
            for r in disliked:
                disliked_tags.update(r['item'].tags)
            reflection_parts.append(
                f"User disliked items with tags: {', '.join(disliked_tags)}. "
                f"Need to avoid these in future."
            )
        
        if not liked and not disliked:
            reflection_parts.append(
                "Neutral feedback. Increasing diversity to explore preferences."
            )
            self.strategy_adjustments['diversity_weight'] = min(
                0.5, self.strategy_adjustments['diversity_weight'] + 0.05
            )
        
        satisfaction = len(liked) / max(len(feedback), 1)
        reflection_parts.append(f"Satisfaction rate: {satisfaction:.0%}")
        
        reflection = " ".join(reflection_parts)
        self.reflections.append(reflection)
        return reflection


# Simulate multi-turn interaction with reflection
reflective_agent = ReflectiveRecAgent(catalog)
user = users[0]
satisfaction_over_turns = []

for turn in range(5):
    print(f"\n{'='*60}")
    print(f"Turn {turn + 1}")
    
    query = "Recommend something for me"
    obs = reflective_agent.observe(user, query)
    plan = reflective_agent.think(obs)
    recs = reflective_agent.act(obs, plan, top_k=5)
    
    print(f"Recommendations:")
    for i, rec in enumerate(recs, 1):
        print(f"  {i}. {rec['item'].name} (score: {rec['score']:.3f})")
    
    # Simulate user feedback (items matching preferences get higher ratings)
    feedback = {}
    prefs = set(user.preferences)
    for rec in recs:
        item = rec['item']
        overlap = len(prefs & set(item.tags + [item.category]))
        base_rating = 2.0 + overlap * 1.0 + np.random.normal(0, 0.5)
        rating = max(1.0, min(5.0, base_rating))
        feedback[item.item_id] = round(rating, 1)
        user.history.append(item.item_id)
    
    reflection = reflective_agent.reflect_on_feedback(recs, feedback)
    print(f"\nReflection: {reflection}")
    
    avg_rating = np.mean(list(feedback.values()))
    satisfaction_over_turns.append(avg_rating)
    print(f"Average rating this turn: {avg_rating:.2f}")

# Plot improvement over turns
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(satisfaction_over_turns) + 1), satisfaction_over_turns, 
        marker='o', linewidth=2, color='steelblue')
ax.set_xlabel('Interaction Turn')
ax.set_ylabel('Average User Rating')
ax.set_title('Agent Self-Improvement via Reflection')
ax.set_ylim(1, 5)
plt.tight_layout()
plt.show()

## 6. Evaluation of LLM Rec Agents

Evaluating LLM agents is more complex than traditional rec systems. We measure:

1. **Recommendation quality**: Relevance, diversity, novelty
2. **Reasoning quality**: Coherence of explanations
3. **Tool usage efficiency**: Number and relevance of tool calls
4. **Adaptability**: Improvement over turns

In [ ]:
def evaluate_agent(agent_class, catalog, users, n_turns: int = 10):
    """Comprehensive evaluation of a rec agent."""
    metrics = {
        'relevance': [],
        'diversity': [],
        'novelty': [],
        'avg_rating': [],
    }
    
    for user in users[:5]:  # Evaluate on 5 users
        agent = agent_class(catalog)
        user_copy = UserProfile(
            user_id=user.user_id, name=user.name,
            preferences=user.preferences, budget=user.budget,
            history=[], ratings={}
        )
        all_recommended = set()
        
        for turn in range(n_turns):
            obs = agent.observe(user_copy, "Recommend something")
            plan = agent.think(obs)
            recs = agent.act(obs, plan, top_k=5)
            
            if not recs:
                continue
            
            # Relevance: tag overlap with preferences
            prefs = set(user_copy.preferences)
            rel_scores = []
            for rec in recs:
                overlap = len(prefs & set(rec['item'].tags + [rec['item'].category]))
                rel_scores.append(overlap / max(len(prefs), 1))
            metrics['relevance'].append(np.mean(rel_scores))
            
            # Diversity: unique categories
            cats = set(r['item'].category for r in recs)
            metrics['diversity'].append(len(cats) / len(recs))
            
            # Novelty: fraction of new items
            new_items = [r for r in recs if r['item'].item_id not in all_recommended]
            metrics['novelty'].append(len(new_items) / len(recs))
            all_recommended.update(r['item'].item_id for r in recs)
            
            # Simulate feedback
            feedback = {}
            for rec in recs:
                item = rec['item']
                overlap = len(prefs & set(item.tags + [item.category]))
                rating = max(1.0, min(5.0, 2.0 + overlap + np.random.normal(0, 0.5)))
                feedback[item.item_id] = round(rating, 1)
                user_copy.history.append(item.item_id)
            
            metrics['avg_rating'].append(np.mean(list(feedback.values())))
            
            if hasattr(agent, 'reflect_on_feedback'):
                agent.reflect_on_feedback(recs, feedback)
            else:
                for item_id, rating in feedback.items():
                    agent.receive_feedback(item_id, rating)
    
    return {k: np.mean(v) for k, v in metrics.items()}


# Compare basic RecAgent vs ReflectiveRecAgent
basic_metrics = evaluate_agent(RecAgent, catalog, users)
reflective_metrics = evaluate_agent(ReflectiveRecAgent, catalog, users)

print("Evaluation Results:")
print(f"{'Metric':<20} {'Basic Agent':<15} {'Reflective Agent':<15}")
print("-" * 50)
for metric in basic_metrics:
    print(f"{metric:<20} {basic_metrics[metric]:<15.3f} {reflective_metrics[metric]:<15.3f}")

# Visualization
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(basic_metrics))
width = 0.35
labels = list(basic_metrics.keys())

ax.bar(x - width/2, list(basic_metrics.values()), width, label='Basic RecAgent', color='steelblue')
ax.bar(x + width/2, list(reflective_metrics.values()), width, label='Reflective RecAgent', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Score')
ax.set_title('Agent Evaluation Comparison')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Summary

| Component | Purpose | Implementation |
|-----------|---------|---------------|
| Planning | Decide recommendation strategy | Rule/prompt-based reasoning |
| Memory (short-term) | Track conversation context | Sliding window of turns |
| Memory (long-term) | Learn user preferences | EMA over interaction history |
| Tool calling | Access structured data | Search, filter, compare, calculate |
| Reflection | Self-improvement | Feedback analysis + strategy adjustment |

**Key references:**
- Wang et al. (2023) - RecAgent: LLM-based recommendation agent
- Huang et al. (2023) - InteRecAgent: interactive recommendation with tools
- Wang et al. (2023) - RecMind: planning-based recommendation
- Park et al. (2023) - Generative Agents: memory and reflection for agents

---

## Exercises

### 🏋️ Exercise 1: Build a Simple LLM Rec Agent with Tool-Calling Simulation

Build a complete agent that handles multi-turn conversations.

In [ ]:
# 🏋️ Exercise 1: Build a conversational rec agent

class ConversationalRecAgent:
    """Agent that handles multi-turn conversations with tool calling."""
    
    def __init__(self, catalog: List[Item], tools: Dict[str, Tool]):
        self.catalog = catalog
        self.tools = tools
        self.memory = ShortTermMemory(max_turns=20)
        self.long_memory = LongTermMemory()
    
    def chat(self, user: UserProfile, message: str) -> str:
        # TODO: Implement the chat method:
        # 1. Parse the user message to determine intent
        # 2. Plan tool calls based on intent
        # 3. Execute tools
        # 4. Generate a natural language response with recommendations
        # 5. Update memory
        pass
    
    def _parse_intent(self, message: str) -> str:
        # TODO: Classify intent: 'recommend', 'compare', 'search', 'feedback'
        pass

# TODO: Test with a multi-turn conversation:
# Turn 1: "What electronics do you recommend?"
# Turn 2: "Can you compare the top 2?"
# Turn 3: "I'll take the first one. What else might I like?"

print("Exercise 1: Implement ConversationalRecAgent")

### 🏋️ Exercise 2: Preference Extraction from Natural Language

In [ ]:
# 🏋️ Exercise 2: Extract preferences from natural language

class PreferenceExtractor:
    """Extract structured preferences from natural language."""
    
    def __init__(self, known_categories: List[str], known_tags: List[str]):
        self.known_categories = [c.lower() for c in known_categories]
        self.known_tags = [t.lower() for t in known_tags]
    
    def extract(self, text: str) -> Dict:
        # TODO: Extract from text like:
        # "I want something portable and smart, preferably electronics under $100"
        # -> {'categories': ['electronics'], 'tags': ['portable', 'smart'], 
        #     'max_price': 100.0, 'sentiment': 'positive'}
        pass

# TODO: Test with various natural language inputs
print("Exercise 2: Implement PreferenceExtractor")

### 🏋️ Exercise 3: Agent with Clarifying Questions

In [ ]:
# 🏋️ Exercise 3: Agent that asks clarifying questions

class ClarifyingRecAgent(RecAgent):
    """Agent that asks questions when user intent is unclear."""
    
    def should_clarify(self, observation: Dict) -> Tuple[bool, str]:
        # TODO: Determine if the query is too vague and generate a clarifying question
        # Return (should_ask, question)
        # E.g., "Something nice" -> (True, "What category are you interested in?")
        pass
    
    def process_with_clarification(self, user: UserProfile, query: str) -> Dict:
        # TODO: If clarification needed, ask. Otherwise, proceed with recommendation.
        pass

print("Exercise 3: Implement ClarifyingRecAgent")